# Task 2: Waiting Time Until Hearing an Owl

## Given parameters

The assigned branch is $\xi_4=3$, with parameters

$$
\xi_5=0.\overline{53}=\frac{53}{99},\qquad
\xi_6=5,\qquad
\xi_7=0.\overline{46}=\frac{46}{99},\qquad
\xi_8=8.
$$

Let $Y$ denote the waiting time until hearing an owl, measured in hours.

The supplied survival function is

$$
S(y)=P(Y>y)
=
\frac{53}{99}e^{-5y^2}
+
\frac{46}{99}e^{-8y^2},
\qquad y\geq0.
$$

The weights satisfy

$$
\frac{53}{99}+\frac{46}{99}=1.
$$

## Objectives

1. Verify that the supplied function defines a valid waiting-time model
2. Calculate the probability of waiting between two and four hours
3. Derive and plot the probability density function
4. Calculate the mean, variance, and quartiles
5. Produce a histogram whose bars represent one-minute interval probabilities
6. Display the summary statistics in the figures and verify the calculations

In [2]:
import sys
from fractions import Fraction
from pathlib import Path

import numpy as np
import scipy
from scipy.integrate import quad
from scipy.optimize import brentq

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# preserve weights
weight_1_exact = Fraction(53, 99)
weight_2_exact = Fraction(46, 99)

assert weight_1_exact + weight_2_exact == 1

weight_1 = float(weight_1_exact)
weight_2 = float(weight_2_exact)

# coefficients in the exponent, with units of hours^(-2).
a = 5.0
b = 8.0

figure_directory = Path("figures")
figure_directory.mkdir(parents=True, exist_ok=True)

print(f"First weight: {weight_1_exact}")
print(f"Second weight: {weight_2_exact}")
print(f"Exact weight sum: {weight_1_exact + weight_2_exact}")

First weight: 53/99
Second weight: 46/99
Exact weight sum: 1


## Model validity and probability density

The survival function satisfies

$$
S(0)=\frac{53}{99}+\frac{46}{99}=1
$$

and

$$
\lim_{y\to\infty}S(y)=0.
$$

For $y\geq0$, both terms are nonnegative and nonincreasing.
Consequently, $0\leq S(y)\leq1$.

The cumulative distribution function is

$$
F(y)=
\begin{cases}
0, & y<0,\\
1-S(y), & y\geq0.
\end{cases}
$$

Differentiating gives the density

$$
f_Y(y)=F'(y)=-S'(y).
$$

Therefore,

$$
f_Y(y)
=
\frac{530}{99}y e^{-5y^2}
+
\frac{736}{99}y e^{-8y^2},
\qquad y\geq0,
$$

and $f_Y(y)=0$ for $y<0$.

The density is nonnegative. Furthermore,

$$
\int_0^\infty f_Y(y)\,dy
=
S(0)-\lim_{y\to\infty}S(y)
=
1.
$$

Thus, the supplied function defines a valid continuous waiting-time model.

In [3]:
def survival(y):
    """p(Y > y), y measured in hours"""
    values = np.asarray(y, dtype=float)
    nonnegative_y = np.maximum(values, 0.0)

    result = (
        weight_1 * np.exp(-a * nonnegative_y**2)
        + weight_2 * np.exp(-b * nonnegative_y**2)
    )

    return np.where(values < 0, 1.0, result)


def cdf(y):
    """p(Y <= y), y in hours"""
    return 1.0 - survival(y)


def density(y):
    """density of Y, in inverse hours"""
    values = np.asarray(y, dtype=float)
    nonnegative_y = np.maximum(values, 0.0)

    result = (
        2 * a * weight_1 * nonnegative_y
        * np.exp(-a * nonnegative_y**2)
        + 2 * b * weight_2 * nonnegative_y
        * np.exp(-b * nonnegative_y**2)
    )

    return np.where(values < 0, 0.0, result)


assert np.isclose(survival(0), 1.0)
assert np.isclose(cdf(0), 0.0)
assert float(density(-1)) == 0.0

print(f"S(0) = {float(survival(0)):.6f}")
print(f"F(0) = {float(cdf(0)):.6f}")
print("Functions defined successfully.")

S(0) = 1.000000
F(0) = 0.000000
Functions defined successfully.


## Probability of waiting between two and four hours

Since Y is continuous, including or excluding the endpoints does notchange the probability.

Using the survival function,

$$
P(2<Y<4)=S(2)-S(4).
$$

Substituting the parameters gives

$$
P(2<Y<4)
=
\frac{53}{99}\left(e^{-20}-e^{-80}\right)
+
\frac{46}{99}\left(e^{-32}-e^{-128}\right).
$$

The survival-function difference is evaluated directly. This avoids subtracting two cumulative probabilities that are both extremely close to one.

In [5]:
probability_2_to_4 = float(survival(2) - survival(4))

assert 0 < probability_2_to_4 < 1

print(f"p(2 < Y < 4) = {probability_2_to_4:.10e}")
print(f"as a percentage = {100 * probability_2_to_4:.10e}%")

p(2 < Y < 4) = 1.1034517630e-09
as a percentage = 1.1034517630e-07%


## Mean and variance

For this nonnegative continuous random variable,

$$
E[Y^r]=\int_0^\infty y^r f_Y(y)\,dy.
$$

Using $f_Y(y)=-S'(y)$ and integrating by parts gives, for $r=1,2$,

$$
E[Y^r]
=
\left[-y^rS(y)\right]_0^\infty
+
r\int_0^\infty y^{r-1}S(y)\,dy.
$$

The boundary terms vanish because the survival function decays exponentially in $y^2$. Hence,

$$
E[Y]=\int_0^\infty S(y)\,dy
$$

and

$$
E[Y^2]=2\int_0^\infty yS(y)\,dy.
$$

For $c>0$, the Gaussian integral gives

$$
\int_0^\infty e^{-cy^2}\,dy
=
\frac{\sqrt{\pi}}{2\sqrt{c}}.
$$

Therefore,

$$
E[Y]
=
\frac{\sqrt{\pi}}{2}
\left(
\frac{53}{99\sqrt{5}}
+
\frac{46}{99\sqrt{8}}
\right).
$$

Also, substituting $u=cy^2$ gives

$$
\int_0^\infty 2y e^{-cy^2}\,dy=\frac{1}{c}.
$$

Consequently,

$$
E[Y^2]
=
\frac{53}{99\cdot5}
+
\frac{46}{99\cdot8}.
$$

Finally,

$$
\operatorname{Var}(Y)=E[Y^2]-E[Y]^2.
$$

The mean has units of hours, while the variance has units of hours squared.

For the waiting time expressed in minutes, $M=60Y$,

$$
E[M]=60E[Y],
\qquad
\operatorname{Var}(M)=60^2\operatorname{Var}(Y).
$$

In [6]:
mean_hours = (np.sqrt(np.pi) / 2 * (weight_1 / np.sqrt(a) + weight_2 / np.sqrt(b)))

second_moment_hours2 = weight_1 / a + weight_2 / b

variance_hours2 = second_moment_hours2 - mean_hours**2

mean_minutes = 60 * mean_hours
variance_minutes2 = 60**2 * variance_hours2

assert variance_hours2 > 0

print(f"Mean = {mean_hours:.9f} hours")
print(f"Mean = {mean_minutes:.6f} minutes")
print(f"Second moment = {second_moment_hours2:.9f} hours^2")
print(f"Variance = {variance_hours2:.9f} hours^2")
print(f"Variance = {variance_minutes2:.6f} minutes^2")

Mean = 0.357765124 hours
Mean = 21.465907 minutes
Second moment = 0.165151515 hours^2
Variance = 0.037155631 hours^2
Variance = 133.760273 minutes^2


## Quartiles

The quartile $q_p$ satisfies

$$
F(q_p)=p,
\qquad p\in\{0.25,0.50,0.75\}.
$$

Equivalently,

$$
\frac{53}{99}e^{-5q_p^2}
+
\frac{46}{99}e^{-8q_p^2}
=
1-p.
$$

The CDF is continuous and strictly increasing for positive waiting times, so each quartile is unique.

The equations are solved numerically using Brent's method on the interval $[0,4]$ hours. For each requested probability, $F(0)-p<0$ and $F(4)-p>0$, roviding a bracket containing the root.

Each solution is checked by substituting it back into the CDF.

In [ ]:
quartile_levels = (0.25, 0.50, 0.75)
quartiles_hours = []

for level in quartile_levels:
    def root_function(y):
        return float(cdf(y)) - level

    assert root_function(0) < 0
    assert root_function(4) > 0

    root = brentq(
        root_function,
        0.0,
        4.0,
        xtol=1e-12,
    )

    assert abs(float(cdf(root)) - level) < 1e-10
    quartiles_hours.append(root)

quartiles_hours = np.array(quartiles_hours)
quartiles_minutes = 60 * quartiles_hours

q1_hours, median_hours, q3_hours = quartiles_hours
q1_minutes, median_minutes, q3_minutes = quartiles_minutes

for label, hours, minutes in zip(
    ("Q1", "Median", "Q3"),
    quartiles_hours,
    quartiles_minutes,
):
    print(f"{label}: {hours:.9f} hours = {minutes:.6f} minutes")